# Troubleshooting Tempotron SNN approaches
## Miranda Gonzales, PhD student 
---
#### Neuroscience Coding Club at UT San Antonio
#### 15 JULY 2026
---
So, I have code that needs troubleshooting. I thought the class-based code (aka method 1) replicated the math & logic of my pure numpy function-based code (aka method 2), but I have proven myself wrong. I started with method 2 and prompted back-and-forth with an LLM to upgrade my function-based code; ultimately, I wanted to improve the evaluation pipeline and make it easier to vary parameters across simulations. While the optimized method 1 runs faster and is easier to evaluate, the two methods are producing different results when given the same inputs.

I'm not sure where they are diverging methematically; please help identify these diversions. I have tried a few things to troubleshoot, but so far the most promising direction was obtained with Gemini; see my summary of the LLM's conclusions in the third section of this notebook. Otherwise, the first section is code that creates input spikes sequences as binary spikes; the second section holds the tempotron codes for both methods and code to run simulations.

-MG

In [ ]:
# Load necessary packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
import json

# MNIST dataset
from sklearn.datasets import fetch_openml
import seaborn as sns
# Note: MNIST dataset is not necessary to run this code; use the manually-designed letter sequence as an alternative to MNIST


In [ ]:
# Helper functions

def get_key(val, dict_roi):
    for key, value in dict_roi.items():
        if val == value:
            if type(key) is str:
                return key
            elif type(key) is int:
                return int(key)
    return "key doesn't exist"

def sldng_wndw_avg(noisy_lst, wndw_size=100):
    data = {'value': noisy_lst}  
    df = pd.DataFrame(data)
    return df['value'].rolling(window=wndw_size).mean()

def load_json(filepath_):
    with open(filepath_, 'r') as json_file:
        loaded_dict_ = json.load(json_file)
    print(f"Dictionary loaded from {file_path}:")
    return(loaded_dict_)
    

### Input Scheme Options:
1) Manually constructed letter sequences
2) MNIST (normalized dataset of hand-written digits)

#### Creating Letter Sequences from Tokenized Patterns

In [ ]:
# Creating input stimulus Firing Patterns for 7 unique tokens (A-G) 

# parameters for input layer
numInputN = 100 # total number of afferent neurons (e.g. inputs)
M_active_afferents = 7 # number of the available afferents eliciting (at least) 1 spike in a given pattern presentation

# temporal parameters for building sequences
dt = 1 # ms
time_per_state = 7 # ms
dt_per_state = int(time_per_state*(1/dt))
t_btwn_state = 0 # ms
nback_int = 4 # number of token states presented in the sequence
Tmax_nback_padded = (time_per_state * nback_int)+(t_btwn_state*(nback_int-1))+50 #50ms includes 10ms pre-stim & 40ms post-stim
t_nback_padded = np.arange(0,Tmax_nback_padded,dt)

# configure randomized set; decide which neurons spike for each token-state
np.random.seed(32) # held at seed = 32 to keep simulations consistent // change to get different input scheme
aff_pooledtotal = [np.random.choice((numInputN), 7*(M_active_afferents),replace=False)][0] #which neuron spikes (for all 7 states)

# Build spiking patterns A-G, assign spike times to each token-state neuron
pattern_times_A = [aff_pooledtotal[:M_active_afferents], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]
pattern_times_B = [aff_pooledtotal[M_active_afferents:(M_active_afferents*2)], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]
pattern_times_C = [aff_pooledtotal[(M_active_afferents*2):(M_active_afferents*3)], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]
pattern_times_D = [aff_pooledtotal[(M_active_afferents*3):(M_active_afferents*4)], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]
pattern_times_E = [aff_pooledtotal[(M_active_afferents*4):(M_active_afferents*5)], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]
pattern_times_F = [aff_pooledtotal[(M_active_afferents*5):(M_active_afferents*6)], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]
pattern_times_G = [aff_pooledtotal[(M_active_afferents*6):], np.random.choice((dt_per_state), (M_active_afferents), replace = False)]

pattern_times = [pattern_times_A, pattern_times_B, pattern_times_C, pattern_times_D, pattern_times_E, pattern_times_F, pattern_times_G]

# Convert neuron-spike time assignments into binary spike trace
spikePatternA = np.zeros((numInputN, dt_per_state))
spikePatternB = np.zeros((numInputN, dt_per_state))
spikePatternC = np.zeros((numInputN, dt_per_state))
spikePatternD = np.zeros((numInputN, dt_per_state))
spikePatternE = np.zeros((numInputN, dt_per_state))
spikePatternF = np.zeros((numInputN, dt_per_state))
spikePatternG = np.zeros((numInputN, dt_per_state))

for j in range(M_active_afferents):
    spikePatternA[pattern_times[0][0][j]][pattern_times[0][1][j]] = 1
    spikePatternB[pattern_times[1][0][j]][pattern_times[1][1][j]] = 1
    spikePatternC[pattern_times[2][0][j]][pattern_times[2][1][j]] = 1
    spikePatternD[pattern_times[3][0][j]][pattern_times[3][1][j]] = 1
    spikePatternE[pattern_times[4][0][j]][pattern_times[4][1][j]] = 1
    spikePatternF[pattern_times[5][0][j]][pattern_times[5][1][j]] = 1
    spikePatternG[pattern_times[6][0][j]][pattern_times[6][1][j]] = 1

dict_spikepatterns_7tokens =   {'A': [spikePatternA, pattern_times_A], 'B': [spikePatternB, pattern_times_B], 
                                'C': [spikePatternC, pattern_times_C], 'D': [spikePatternD, pattern_times_D], 
                                'E': [spikePatternE, pattern_times_E], 'F': [spikePatternF, pattern_times_F], 
                                'G': [spikePatternG, pattern_times_G]}

# print key tokens and their [neuron identities, and spike times]
print([[i,dict_spikepatterns_7tokens[i][1]] for i in dict_spikepatterns_7tokens.keys()])

In [ ]:
# function to build input spike trains for a given sequence of states A-G
    
def build_inputspikes_general(memory_seq, n_in, dict_patterns, time_params, if_plot=False, hint_type=None):
    '''
    memory_seq = nback_seq
    n_in = numInputN
    dict_patterns = dict_spikepatterns_7tokens
    time_params = [Tmax_nback_padded, t_btwn_state, time_per_state, t_prestim, dt]
    plot option, boolean
    hint_type = None (default) | valid inputs: 'hint_1'
    '''
     
    if hint_type == 'hint_1': # spike if there's a vowel
        n_total = n_in+1
    else:
        n_total = n_in
        
    dt_seq = int(1/time_params[4])
    input_spikes = np.zeros((n_total, time_params[0]*dt_seq))
    
    for s, state in enumerate(memory_seq): 
        beg_i = (((time_params[1]+time_params[2])*(s)) + time_params[3])*dt_seq
        end_i = (beg_i + time_params[2])*dt_seq
        time_slice = np.arange(beg_i, end_i)
        
        for neuron in range(n_in):
            input_spikes[neuron,time_slice] = dict_patterns[state][0][neuron]
      
        if hint_type == 'hint_1':
            # add spikes at hint neuron for vowels
            if state == 'A' or state == 'E':
                input_spikes[n_in+0,time_slice] = np.array([1,0,1,0,1,0,1])

        

    # test to make sure that spike patterns saved accurately, updated for n7back memory
    if if_plot == True:
        f_raster = lambda x: np.where(x == 1)
        
        raster_pos = []
        for n, neuron in enumerate(range(n_total)):
            raster_pos.append(f_raster(input_spikes[n])[0])
        
        ax = plt.subplot(1, 1, 1)
        plt.eventplot(raster_pos)
        plt.xlabel('Time (ms)')
        plt.ylabel('Afferent Neuron')
        flatseq = ''
        for s in memory_seq:
            flatseq += s
        plt.title('Raster Plot of '+str(len(memory_seq))+'-state History: '+flatseq+'\nHint type: '+str(hint_type))
        ax.set_xlim([0,time_params[0]])
        #ax.set_ylim(99,104)
        #plt.savefig(save_text_260615+'.png')
        plt.show()
        
    return input_spikes.T # output shape = (t_max, n_in+4)

In [ ]:
# try function to get input spikes and plot raster
try_n6back_seq = 'CABBAGE'
for hint_i in [None, 'hint_1']:
    #hint_ii = 'None' if hint_i==None else hint_i
    try_spikes = build_inputspikes_general(try_n6back_seq, numInputN, dict_spikepatterns_7tokens, 
                                      [Tmax_nback_padded, t_btwn_state, time_per_state, 10, 1], if_plot=True, hint_type=hint_i)
try_n4back_seq = 'BADD'
for hint_i in [None, 'hint_1']:
    #hint_ii = 'None' if hint_i==None else hint_i
    try_spikes = build_inputspikes_general(try_n4back_seq, numInputN, dict_spikepatterns_7tokens, 
                                      [Tmax_nback_padded, t_btwn_state, time_per_state, 10, 1], if_plot=True, hint_type=hint_i)


In [ ]:
# Set mock-training dataset

df_train_options = {'seqs': ['GABB', 'FEED', 'EFFA', 'CAGE', 'BADG', 'DECA', 'DEBA'], 
                    'possible_labels':[['A'], ['B'], ['C'], ['D'], ['E'], ['F'], ['G']]}

# GOAL: SEE 4 LETTERS, PREDICT THE 5TH! (in a language where only these 7 letters exist:)
### GABBA, where tempotron predicts A as the 5th letter 
# FEEDBAG, where tempotron predicts B as the 5th letter 
## EFFACE, where tempotron predicts C as the 5th letter 
### CAGED, where tempotron predicts D as the 5th letter 
### BADGE, where tempotron predicts E as the 5th letter 
### DECAF, where tempotron predicts F as the 5th letter
### DEBAG, where tempotron predicts G as the 5th letter 

# add more words (or nonsense), if you feel like it
# Note: a sequence can have multiple labels.
#### example: for the words 'cabbage' and 'cabbed' the same 4-letters 'CABB' have 2 possible labels of ['A', 'E']

df_train_1k = {'seqs':[], 'possible_labels':[]}
np.random.seed(32) # remove if you want a random training sequence every time
randint_1k = np.random.choice((range(len(df_train_options['seqs']))), 1000, replace=True)
#print(len(randint_1k), randint_1k[:10])

for i in randint_1k:
    df_train_1k['seqs'].append(df_train_options['seqs'][i])
    df_train_1k['possible_labels'].append(df_train_options['possible_labels'][i])
#print(df_train_1k['seqs'][:10])
#print(df_train_1k['possible_labels'][:10])

#### Importing MNIST dataset and Creating Inputs using TTFS

In [ ]:
# Prep MNIST input arrays

# Load MNIST dataset from OpenML
mnist = fetch_openml('mnist_784', version=1)

# Convert the data into numpy arrays
x_mnist = mnist.data.to_numpy() # features (pixel values; grayscale)
y_mnist = mnist.target.to_numpy() # labels

# Convert data to the correct type
x_mnist = x_mnist.astype(np.float32)
y_mnist = y_mnist.astype(np.int64)

print(x_mnist.shape, x_mnist[0]) # shape is 70k samples, 784 pixels per sample
print(y_mnist.shape, y_mnist[0]) # shape is 70k samples

x_mnist_train = x_mnist[:50000]
y_mnist_train = y_mnist[:50000]
x_mnist_labelval = x_mnist[50000:60000]
y_mnist_labelval = y_mnist[50000:60000]
x_mnist_test = x_mnist[60000:]
y_mnist_test = y_mnist[60000:]

    

In [ ]:
# Plotting functions for MNIST inputs

def plot_mnist_trial(x,y):
    # Create the subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
    
    # Plot the histogram on the first subplot
    x_reshape = x.reshape(28,28)
    ax1.imshow(x_reshape, cmap='gray') #hist(data, bins=30)
    ax1.set_title('Label: '+str(y))
    
    # Plot a line plot on the second subplot
    ax2.hist(x, bins=30)
    ax2.set_title('Histogram: Pixel values')
    ax2.set_xlabel('Pixel Value')
    ax2.set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

def plot_ttfs_raster(spike_times_2d, label):
    spike_times_flat = spike_times_2d.flatten()
    neuron_indices = np.arange(len(spike_times_flat))
    
    # Only keep neurons that spike
    valid = ~np.isinf(spike_times_flat)
    spike_times_flat = spike_times_flat[valid]
    total_active = int((len(spike_times_flat)/len(valid)) *100)
    neuron_indices = neuron_indices[valid]
    
    # Prepare data for eventplot: a list of spike time(s) per neuron
    events = [[] for _ in range(neuron_indices.max() + 1)]
    for idx, t in zip(neuron_indices, spike_times_flat):
        events[idx].append(t)
    
    # Plot raster
    plt.figure(figsize=(10, 6))
    plt.eventplot(events, colors='black', linelengths=1.5)
    plt.xlabel("Time (ms)")
    plt.ylabel("Pixel (flattened index)")
    plt.title("Raster Plot of TTFS-Encoded MNIST Input\nActive Neurons = "+str(total_active)+"% | Label = "+str(label)+" | Note: brighter pixels spike earlier")
    plt.tight_layout()
    plt.show()

In [ ]:
# Function to create spike times of input neurons from MNIST digits using Time-to-first spike (TTFS) scheme (plotting optional)

def input_spikes_mnist(x, y, t_min, t_max, threshold=0.0, if_plot=False):
    """
    Input parameters:
    x (np.array) : pixel values (grascale) 784 pixels per MNIST image
    y (int) : correct label assingment for MNIST image
    t_min : earliest possible spike time
    t_max : latest possible spike time
    threshold : pixel intensity below which no spike is generated (optional)
    if_plot (boolean; True or False) : will plot the spiking raster if True

    Returns:
    spike_times : 2D array of same shape as image, with spike times (inf if no spike)
    optional : 2 plots
    """
    
    norm_image = (x - x.min()) / (x.max() - x.min()) 
    spike_times = t_min + (1 - norm_image) * (t_max - t_min)
    spike_times[norm_image <= threshold] = np.inf  # no spike

    if if_plot == True:
        plot_mnist_trial(x, y)
        plot_ttfs_raster(spike_times, y)

    input_spikes = np.zeros((t_max, len(x))) # len(x)=number of input neurons
    for i in range(len(x)):
        t = spike_times[i]
        if t < t_max:
            if round(t)==t_max:
                offset=1
            else:
                offset=0
            input_spikes[round(t)-offset, i] = 1.0  # spike at first-available time
        
    return input_spikes 


In [ ]:
# Function to plot heatmap of MNIST pixel parameters (i.e., weights)

def plot_mnist_heatmap(weights_1d, title="Synaptic Weights to One Output Neuron", savefig=False, savename="weight_heatmap.pdf"):
    """
    Plots a 28x28 heatmap for weights from 784 input neurons to one output neuron.
    
    Parameters:
        weights_1d (array-like): A 1D array of 784 weights.
        title (str): Title of the plot.
    """
    assert len(weights_1d) == 784, "Input weight array must have 784 elements."
    
    weights_2d = np.reshape(weights_1d, (28, 28))
    
    plt.figure(figsize=(6, 6))
    sns.heatmap(weights_2d, cmap='viridis', cbar_kws={'label': 'Weight'}, square=True)
    plt.title(title)
    plt.axis('off')  # Optional: hide axis for cleaner image view
    plt.tight_layout()
    if savefig == True:
        plt.savefig(savename)
    plt.show()

In [ ]:
# Example code to create MNIST plots
ex_MNIST_spl = 7
spike_trains = input_spikes_mnist(x_mnist_train[ex_MNIST_spl], 
                                  y_mnist_train[ex_MNIST_spl], 
                                  t_min=0, t_max=30, 
                                  threshold=0.0, 
                                  if_plot=True)

## Tempotron Code

### Tempotron Basics: PSP Voltage as a Kernel Operation

In [ ]:
# Tau (membrane time constant) demonstration

# Relevant neuron model & simulation parameters
v0 = 2.12
dt = 1
time_per_state = 7
Tmax_padded = (time_per_state * 2)+110
t_padded = np.arange(0, Tmax_padded, dt)

# Example 1 of tau (to be plotted in black)
tau = 15 
tau_s = tau/4
kernelT_padded = v0 * (np.exp(-t_padded/tau) - np.exp(-t_padded/tau_s))

# Example 2 of tau (to be plotted in red)
tau_larger = 10 
tau_s_larger = tau_larger/4
kernelT_largertau = v0 * (np.exp(-t_padded/tau_larger) - np.exp(-t_padded/tau_s_larger))

# Example 3 - simpler LIF eq, single tau (to be plotted in blue)
tau_single = 2
kernerT_singletau = v0 * np.exp(-t_padded/tau_single)

# Add lines emulating voltage trace to the plot & show
show_tstep = 2
plt.plot(t_padded, kernelT_padded, linestyle = '--', color = 'k')
plt.plot(t_padded, kernelT_largertau, linestyle = '-', color='r')
plt.plot(t_padded, kernerT_singletau, linestyle = '-', color='blue')
plt.plot(t_padded[show_tstep], kernelT_padded[show_tstep], marker ='.', color = 'k')
plt.plot(t_padded[show_tstep], kernelT_largertau[show_tstep], marker ='.', color = 'r')
plt.plot(t_padded[show_tstep], kernerT_singletau[show_tstep], marker ='.', color = 'blue')

plt.title('LIF & tau (membrane time constant) Demonstration')
plt.show()

### Tempotron Method 1: Class-based Approach

In [ ]:
# Define Tempotron Neuron model as a pythonic class (updated approach, July 2026; pending evaluation)
# ---------
### Note MG: for original problematic code (TempotronNetwork_old & TempotronNeruon_old), see cell at end of notebook with first line:
### # Gemini evaluation of Method 1:

class TempotronNeuron:
    def __init__(self, n_inputs, w_import=False, w_mean=0.001, w_stdev=0.0001, tau=15.0, tau_s=3.75, v0=2.12, v_thresh=1.0, lr=0.001, label=''):
        self.n_inputs = n_inputs
        self.tau = tau
        self.tau_s = tau_s
        self.v0 = v0
        self.v_thresh = v_thresh
        self.lr = lr
        self.label = 'Tempotron_'+label
        self.reset_state()
        if np.isnan(w_import).all(): 
            self.weights = np.random.normal(w_mean, w_stdev, n_inputs)
        else:
            self.weights = w_import
            

    def reset_state(self):
        self.V_peak = 0.0
        self.V_trace = None

    def PSP_kernel(self, t_diff):
        """Post-synaptic potential kernel"""
        k = self.v0 * (np.exp(-t_diff / self.tau) - np.exp(-t_diff / self.tau_s))
        k[t_diff < 0] = 0
        return k

    def compute_voltage(self, spike_trains, time_window):
        all_K = np.zeros((self.n_inputs, len(time_window)))   

        # Pre-compute the continuous kernel array over the time window (updated approach, July 2026)
        kernel_array = self.PSP_kernel(time_window)
        
        for i in range(self.n_inputs):
            spikes = spike_trains[i]
            if np.sum(spikes) == 0:
                continue 
            # old approach, pre-July 2026 update    
            #t_diffs = np.convolve(spikes, time_window)[:len(time_window)] 
            #all_K[i] = self.PSP_kernel(t_diffs)

            # Convolve the binary spikes with the pre-computed kernel array (updated approach, July 2026)
            all_K[i] = np.convolve(spikes, kernel_array)[:len(time_window)]
        
        V = np.dot(self.weights, all_K)
        self.V_trace = V
        self.V_peak = np.max(V)
        return V

    def fire(self):
        return self.V_peak >= self.v_thresh

    def update_weights(self, spike_trains, time_window, target_output):
        pred = self.fire()
        if pred == target_output:
            return 0  # No error

        # Isolate the exact time index and value of the maximum voltage
        t_max = time_window[np.argmax(self.V_trace)]
        
        for i in range(self.n_inputs):
            spikes = spike_trains[i]
            if np.sum(spikes) == 0:
                continue

            # Map spike indices back to the actual time array to handle any dt resolution (updated approach, July 2026)
            spike_indices = np.where(spikes > 0)[0]
            spike_times = time_window[spike_indices]
            
            t_diffs = t_max - spike_times #t_diffs = t_max - np.array(np.where(spikes>0))
            valid_diffs = t_diffs[t_diffs >= 0]
            
            if valid_diffs.size == 0:
                continue
            dV_dw = np.sum(self.PSP_kernel(valid_diffs))
            delta_w = self.lr * dV_dw
            
            if pred and not target_output:
                self.weights[i] -= delta_w
            elif not pred and target_output:
                self.weights[i] += delta_w
        
        self.reset_state()
        return 1  # Error occurred

In [ ]:
# Define Tempotron Network layer as a pythonic class (unchanged)

class TempotronNetwork:
    def __init__(self, n_neurons, n_inputs, w_import_all, time_window, tempo_labels, **neuron_params):
        if np.isnan(w_import_all).all():
            self.neurons = [TempotronNeuron(n_inputs, w_import=[np.nan], label=tempo_labels[_], **neuron_params) for _ in range(n_neurons)]
        else:
            self.neurons = [TempotronNeuron(n_inputs, w_import=w_import_all[_], label=tempo_labels[_], **neuron_params) for _ in range(n_neurons)]
        self.time_window = time_window
        self.error_log = []
        self.error_type_log = []
        self.true_acc_log = []
        self.all_targets = []
        for i in self.neurons:
            self.error_type_log.append([])
            self.true_acc_log.append([])
            self.all_targets.append([])

    def reset(self):
        for neuron in self.neurons:
            neuron.reset_state()

    def train_step(self, spike_trains, target_outputs, metric_tracker=None):
        errors = []
        for n, neuron, target in zip(range(len(self.neurons)), self.neurons, target_outputs):
            #print(neuron.label, target)
            self.reset()
            neuron.compute_voltage(spike_trains, self.time_window)
            error = neuron.update_weights(spike_trains, self.time_window, target)
            errors.append(error)
            if metric_tracker != None:
                metric_tracker.update(n, error, target)
        self.error_log.append(np.sum(errors))
        for e, error in enumerate(errors):
            self.all_targets[e].append(target_outputs[e])
            self.error_type_log[e].append(error)
            self.true_acc_log[e].append(0)
        if np.sum(errors) == 0:
            if len(np.where(target_outputs > 0)) > 1:
                raise CustomError("Target output had more correct values than expected; see train_step function in TempotronNetwork.")
            if len(target_outputs) == 1:
                self.true_acc_log[0][-1] = 1
            else:
                self.true_acc_log[np.where(np.array(target_outputs) > 0)[0][0]][-1] = 1
            
        self.reset()

    def predict(self, spike_trains):
        preds = []
        for neuron in self.neurons:
            neuron.compute_voltage(spike_trains, self.time_window)
            preds.append(int(neuron.fire()))
        self.reset()
        return preds

In [ ]:
# Define Tempotron Metric tracker to evaluate training progress (unchanged)
class TempotronMetrics:
    def __init__(self, n_classes):
        self.n_classes = n_classes
        self.TP = np.zeros(n_classes) # True positive
        self.TN = np.zeros(n_classes) # True negative
        self.FP = np.zeros(n_classes) # False positive
        self.FN = np.zeros(n_classes) # False negative

    def update(self, tempotron_idx, is_error, is_target):
        if not is_error and is_target:
            self.TP[tempotron_idx] += 1
        elif not is_error and not is_target:
            self.TN[tempotron_idx] += 1
        elif is_error and not is_target:
            self.FP[tempotron_idx] += 1
        elif is_error and is_target:
            self.FN[tempotron_idx] += 1

    def reset(self):
        self.TP[:] = 0
        self.TN[:] = 0
        self.FP[:] = 0
        self.FN[:] = 0

    def compute_confusion_metrics(self):
        metrics = {}
        for k in range(self.n_classes):
            tp, tn, fp, fn = self.TP[k], self.TN[k], self.FP[k], self.FN[k]
            
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # true positive rate
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # true negative rate
            precision   = tp / (tp + fp) if (tp + fp) > 0 else 0  # of fires, how many correct
            f1          = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0
    
            metrics[k] = {
                'sensitivity': sensitivity,  # is it detecting its class?
                'specificity': specificity,  # is it staying silent when it should?
                'precision':   precision,
                'f1':          f1
            }
        return metrics

    def evaluate_system(self, layer, predictions, y_dataset):
        # 7x7 matrix: rows = true class, cols = predicted class
        confusion_pure = np.zeros((self.n_classes, self.n_classes), dtype=int)
        confusion_all = np.zeros((self.n_classes, self.n_classes), dtype=int)
        no_decision = 0    # no tempotron fired
        ambiguous   = 0    # multiple tempotrons fired
    
        for tt, true_class in enumerate(y_dataset):
            fired = predictions[tt]
            
            if np.sum(fired) == 1:
                confusion_pure[list(true_class).index(1), fired.index(1)] += 1
                confusion_all[list(true_class).index(1), fired.index(1)] += 1
            elif np.sum(fired) == 0:
                no_decision += 1
            else:
                ambiguous += 1
                for i in np.where(np.array(fired)==1)[0]:
                    confusion_all[list(true_class).index(1), i] += 1
                    
    
        return confusion_pure, confusion_all, no_decision, ambiguous


    def plot_confusion_matrix(self, confusion, no_decision, ambiguous, epoch=None, class_names=None, normalize=True, if_savefig=False, add_notes=''):
        """
        Plot confusion matrix as a heatmap.
        
        Args:
            confusion:    (n_classes x n_classes) numpy array from evaluate_system()
            epoch:        int or string, used in plot title (optional)
            class_names:  list of class label strings e.g. ['A','B','C',...] (optional)
            normalize:    if True, show row-normalized proportions alongside raw counts
        """
        n = self.n_classes
    
        if class_names is None:
            class_names = [f'Class {i}' for i in range(n)]
    
        if normalize:
            row_sums = confusion.sum(axis=1, keepdims=True)
            # avoid divide by zero for empty rows
            norm_confusion = np.where(row_sums > 0, confusion / row_sums, 0)
        else:
            norm_confusion = confusion
    
        fig, ax = plt.subplots(figsize=(8, 6))
    
        im = ax.imshow(norm_confusion, interpolation='nearest', 
                       cmap='gist_heat_r', vmin=0, vmax=np.max(norm_confusion))
    
        # colorbar
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Proportion' if normalize else 'Count', fontsize=11)
    
        # axis labels and ticks
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
        ax.set_yticklabels(class_names, fontsize=10)
        ax.set_xlabel('Predicted Class', fontsize=12)
        ax.set_ylabel('True Class', fontsize=12)
    
        title = 'Confusion Matrix ('+add_notes+')\nNo decision = '+str(no_decision)+', Ambiguous = '+str(ambiguous)
        if epoch is not None:
            title += f' — Epoch {epoch}'
        ax.set_title(title, fontsize=13, pad=12)
    
        # annotate each cell with both raw count and proportion
        for i in range(n):
            for j in range(n):
                raw   = int(confusion[i, j])
                prop  = norm_confusion[i, j]
                # use white text on dark cells for readability
                color = 'white' if prop > (0.2*np.max(norm_confusion)) else 'black'
                ax.text(j, i, f'{raw}\n({prop:.2f})',
                        ha='center', va='center',
                        fontsize=9, color=color)
    
        # highlight diagonal in a distinct edge color
        for i in range(n):
            ax.add_patch(plt.Rectangle(
                (i - 0.5, i - 0.5), 1, 1,
                fill=False, edgecolor='magenta', linewidth=2
            ))
    
        plt.tight_layout()
        if if_savefig==True:
            plt.savefig('ConfusionMatrix_'+add_notes+'_epoch'+str(epoch)+'_'+date_info+'.pdf')
        plt.show()
        

### Tempotron Method 2: Function-based Approach

In [ ]:
#### clunky training and evaluation functions adapted for tau customization
# Note MG: need to update to (1) update weights as a function of all recent spikes and (2) account for dt != 1

def calcError_allseqs_tau(tempotron, weights, nn_input, try_Tmax, _tau=15.0, t_hint=None): 
    # setting evaluation dataset
    evaluation_seqs = df_train_options['seqs']
    evaluation_labels = df_train_options['possible_labels']

    # time series info
    if t_hint == 'hint_1':
        nn_input_hints = nn_input+1
    else:
        nn_input_hints = nn_input
    t_nback_padded = np.arange(0, try_Tmax, 1) # dt fixed at 1
    v0 =  2.12
    _tau_s = _tau/4.0
    kernelT_padded = v0 * (np.exp(-t_nback_padded/_tau) - np.exp(-t_nback_padded/_tau_s))
    
    # setting theoretical limits (6back memory)
    error_n = 0 
    type_template = {'TP':0, 'TN':0, 'FP':0, 'FN':0, 'stochastic_P':0, 'stochastic_N':0}
    count_outcome_type = {'all':{}} 
    for _key1 in count_outcome_type.keys():
        for _key2 in type_template.keys():
            count_outcome_type[_key1][_key2] = 0
    
    for a,b in enumerate(evaluation_seqs):
        pos_type = 'all' 
        if len(evaluation_labels[a]) == 1:
            pattern_key = 1 if evaluation_labels[a][0] == tempotron else -1
        else:
            pattern_key = 0 
        spikePattern = build_inputspikes_general(b, nn_input, dict_spikepatterns_7tokens, 
                                                [try_Tmax, t_btwn_state, time_per_state, 10, 1], if_plot=False, hint_type=t_hint)
        spikePatternT = spikePattern.copy().T
        
        K = np.zeros((nn_input_hints, t_nback_padded.size)) 
    
        for j in range(nn_input_hints):
            K[j,:] = np.convolve(spikePatternT[j, :], kernelT_padded)[:t_nback_padded.size]
    
        tempotronPSP_ind = np.dot(weights, K)[0] 
        idx_max = np.argmax(tempotronPSP_ind)
        v_max = tempotronPSP_ind[idx_max]
    
        if ( v_max > 1 and pattern_key < 0 ):
            error_n += 1
            count_outcome_type[pos_type]['FP'] += 1
        elif ( v_max < 1 and pattern_key > 0 ):
            error_n += 1
            count_outcome_type[pos_type]['FN'] += 1
        elif ( v_max < 1 and pattern_key < 0 ):
            count_outcome_type[pos_type]['TN'] += 1
        elif ( v_max > 1 and pattern_key > 0 ):
            count_outcome_type[pos_type]['TP'] += 1
        elif ( v_max > 1 and pattern_key == 0 ):
            count_outcome_type[pos_type]['stochastic_P'] += 1
            if tempotron=='G' or tempotron=='C' or tempotron=='F':
                error_n += 1
        elif ( v_max < 1 and pattern_key == 0 ):
            count_outcome_type[pos_type]['stochastic_N'] += 1
        else:
            raise ValueError('i dont even know lol')

    error_prop = error_n/len(evaluation_seqs)
    return error_prop, count_outcome_type

def train_nback_seqorder_tau(tempotron, n_input, try_Tmax, _tau=15.0, t_hint=None, numTrainingSteps=1000, try_seed=32): #prev. function input set-up: ...(numBatches, target, training_style)
    training_seqorder = df_train_1k['seqs']
    training_labelorder = df_train_1k['possible_labels']

    if t_hint == 'hint_1':
        n_input_hints = n_input+1
    else:
        n_input_hints = n_input
    learningRate = 0.005

    # Initialize weights
    np.random.seed(try_seed) 
    weights = np.random.normal(loc=0.1, scale=(0.025), size= (1, n_input_hints))

    # time-series info
    t_nback_padded = np.arange(0, try_Tmax, 1)
    v0  = 2.12
    _tau_s = _tau/4.0
    kernelT_padded = v0 * (np.exp(-t_nback_padded/_tau) - np.exp(-t_nback_padded/_tau_s))
    
    # Markers for training efficiency
    weights_tracker = {} 
    error_progression = np.zeros((numTrainingSteps+1))
    otype_template = {'TP':0, 'TN':0, 'FP':0, 'FN':0, 'stochastic_P':0, 'stochastic_N':0}
    outcome_type_count = {'all':{}} 
    for _key1 in ['all']: 
        for _key2 in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
            outcome_type_count[_key1][_key2] = np.zeros((numTrainingSteps+1))
    
    error_progression[0], current_outcome = calcError_allseqs_tau(tempotron, weights, n_input, try_Tmax, _tau, t_hint) 
    for _key1 in ['all']: 
        for _key2 in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
            outcome_type_count[_key1][_key2][0] = current_outcome[_key1][_key2]

    
    for i in range(numTrainingSteps):
        #weights_tracker[i] = weights
        
        #initialize K; will be used later to calculate tempotron response to spike pattern
        K = np.zeros((n_input_hints, try_Tmax)) 

        #select pair from training sequence & assign true label
        seqN = training_seqorder[i]      
        labelN = training_labelorder[i]

        #grab/call appropriate spike pattern (i.e., raster plot) & assign +/- label
        #### output shape = (t_max, n_in+4)
        spikePattern = build_inputspikes_general(seqN, n_input, dict_spikepatterns_7tokens, 
                                                [try_Tmax, t_btwn_state, time_per_state, 10, 1], if_plot=False, hint_type=t_hint)
        spikePatternT = spikePattern.copy().T

        if labelN == tempotron:
            pattern_key = 1
            
        else:
            pattern_key = -1
        
        for j in range(n_input_hints):        
            K[j,:] =  np.convolve(spikePatternT[j, :], kernelT_padded)[:t_nback_padded.size] # NOTE: will be weird output if dt is not 1
        tempotronPSP = np.dot(weights, K)[0]
        idx_max = np.argmax(tempotronPSP)
        v_max = tempotronPSP[idx_max]

    
        if ( v_max > 1 and pattern_key < 0 ) or ( v_max < 1 and pattern_key > 0 ):
                #enters if loop when: (1) fires and was not supposed to OR (2) did not fire and was supposed to
            wi_update = np.zeros(n_input_hints)
            t_last_spike = np.zeros((n_input_hints))
            
            for t in t_nback_padded[:idx_max]: #slicing basic time list up to the point in time of the vmax 
                for k in range(n_input_hints): #now applying as an iteration thru all input cells
                    if spikePatternT[k,t]==1: #finally, update the last_spike 'counter'/'place-marker' with the current/'t+1' time-point
                        t_last_spike[k] = t+1 
            
            for k in range(n_input_hints):
                if t_last_spike[k] > 0:
                    diff = t_nback_padded[idx_max] -  (t_last_spike[k]-1) 
                    
                    if diff > 0:
                        wi_update[k] += v0 * (np.exp(-diff/_tau) - np.exp(-diff/_tau_s)) #the unique t_diff's related to each aff_n's spike index means the weight update is unique, but predictable for each 
            
            if pattern_key < 0:
                weights -= learningRate*wi_update
            elif pattern_key > 0:
                weights += learningRate*wi_update
                
        
        error_progression[i+1], current_outcome  = calcError_allseqs_tau(tempotron, weights, n_input, try_Tmax, _tau, t_hint) 
        if (i+1) % 100 == 0:
            print(tempotron, i, error_progression[i+1], current_outcome)
       
        for hh in ['all']: 
            for jj in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
                #print(hh, jj)
                outcome_type_count[hh][jj][i+1] = current_outcome[hh][jj]

        if error_progression[i+1] <= 0.10:
            weights_tracker[i] = weights

        if error_progression[i+1] == 0.0000 and error_progression[i] == 0.0000:
            return weights, weights_tracker, error_progression, outcome_type_count

    return weights, weights_tracker, error_progression, outcome_type_count

In [ ]:
# summary plot functions - combining error progressions for all 7 tempotrons   
tempotron_options = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

def plt_error_training(plt_error, avg_window=100, title_notes='', savefig=False, savename=''):
    plty_etracker = {}
    for i, lst_i in enumerate(plt_error):   
        pltx = len(lst_i)
        plty_etracker[str(tempotron_options[i]+'_sliding_window_avg')] = sldng_wndw_avg(lst_i, wndw_size=avg_window)
        
        plt.plot(list(range(pltx)), plty_etracker[str(tempotron_options[i]+'_sliding_window_avg')], linewidth = 1, alpha = 1)#, zorder=zorder_lst[i])#, color=color_lst[i])
    
    plt.xlim([0,pltx])
    plt.xlabel('Training Step Number')
    plt.ylabel('Error')
    #plt.axhline(y=0, color='gray', linestyle=':')
    #plt.axhline(y=(8/80), color='gray', linestyle=':')
    plt.legend(labels = tempotron_options)
    plt.suptitle("Error Progression | n6back Training for all 7 Tempotrons"+title_notes)# \ntau = "+str(tau)+" | seed = "+str(seed)+" | learning rate = "+str(learningRate))
    if savefig == True:
        plt.savefig('TempotronNCC_Errorplot_'+savename+'.pdf')
    plt.show()
        


### Running Simulations for **Class-based Method** using *MNIST inputs (TTFS scheme)*

In [ ]:
# Training over pre-defined number of samples (option to train continuously across many epochs)

# update every time...will be used for plotting & saving data appropriately
date_info = "260714"

# Initialize basic parameters of interest
lr_combo = {'combo110': [110, 0.001]} # learning rate
w_m = 0.01                            # mean of weight initialization (from normal dist.)
w_sd = 0.001                          # standard deviation of weight initialization (from normal dist.)

# Define time window for each MNIST sample simulation
t_enc = 31 # ms
time_window = np.arange(0,t_enc,1)  

# Define number of epochs, total number of training samples, and how many often the network will be evaluated
epochs = 1 #2 # each epoch trains on all 1k training spls
n_train_spls = 1000 #50000*epochs # epoch max = 50000
n_eval_slice = 100 #2500

# Initialize dictionary to save final weights
dict_final_w = {}
dict_train_perform = {}

# For loop set-up to try out different learning rates associated with a unique experiment parameter-combo ID
for cx in lr_combo:
    param_combo = lr_combo[cx][0] 
    lr = lr_combo[cx][1] 
    param_id = 'w_m=0.01, w_sd=0.001, lr='+str(lr)

    # Add dictionary to save final weights for this loop's lr_combo
    dict_final_w['combo'+str(param_combo)] = {'Tempo0':[], 'Tempo1':[], 'Tempo2':[], 'Tempo3':[], 'Tempo4':[], 
                                              'Tempo5':[], 'Tempo6':[], 'Tempo7':[], 'Tempo8':[], 'Tempo9':[]}
    add_tempolabels = [str(i) for i in range(10)]
    print(add_tempolabels)
    
    # Optional: Import previous final weights & reformat in np.array (10x784)
    #w_nparray = np.zeros((10,784))
    #for i in range(10):
    #    w_nparray[i] = np.array(dict_w_import_t2[cx]['Tempo'+str(i)])
    
    # Create a Tempotron network with 10 neurons and 784 input channels
    tempotron_net = TempotronNetwork(n_neurons=10, n_inputs=784, w_import_all=np.nan, w_mean=w_m, w_stdev=w_sd, time_window=time_window, tempo_labels=add_tempolabels, lr=lr)
        # Note: to initialize weights from w_mean & w_stdev, use: w_import_all=np.nan
    

    # Initialize arrays to store evaluation metrics
    all_train_acc = np.zeros((((n_train_spls//n_eval_slice)+1)))
    all_total_slices = np.zeros((((n_train_spls//n_eval_slice)+1)))
    all_eval_slices = np.zeros((((n_train_spls//n_eval_slice)+1),10))
    x_lst_eval = np.zeros((((n_train_spls//n_eval_slice)+1),10))
    
    ###
    # Begin Training
    ###
    for n_eval_epoch in range(n_train_spls):
        # Resets sample ID back to 0 for every epoch
        n_eval = n_eval_epoch % 50000
        
        # Generate spike trains & target output for single MNIST digit
        spike_trains = input_spikes_mnist(x_mnist_train[n_eval], y_mnist_train[n_eval], t_min=0, t_max=t_enc, threshold=0.0, if_plot=False)
        spike_trains_T = spike_trains.T
        target_outputs = np.zeros(10)
        target_outputs[y_mnist_train[n_eval]] = 1
        
        # Train step
        tempotron_net.train_step(spike_trains_T, target_outputs)

        # Evaluation
        if (n_eval+1) % n_eval_slice == 0:
            x_lst_eval[(n_eval_epoch//n_eval_slice)+1][:] = n_eval_epoch
            all_train_acc[(n_eval_epoch//n_eval_slice)+1] = np.sum(tempotron_net.true_acc_log)/len(tempotron_net.true_acc_log[0])
            all_total_slices[(n_eval_epoch//n_eval_slice)+1] = np.sum([i[-n_eval_slice:] for i in tempotron_net.true_acc_log])/n_eval_slice
            
            rec_acc = np.sum([i[-n_eval_slice:] for i in tempotron_net.true_acc_log], axis=1)
            rec_tot = np.sum([i[-n_eval_slice:] for i in tempotron_net.all_targets], axis=1)
            all_eval_slices[(n_eval_epoch//n_eval_slice)+1] = rec_acc/rec_tot
    ###
    # End of Training
    ###
    
    # Save & plot accuracy vs training progression 
    dict_train_perform['combo'+str(param_combo)] = {'true_acc_log': tempotron_net.true_acc_log, 
                                                    'all_targets': tempotron_net.all_targets}

    # Plot & Save Accuracy across training
    plt.figure(figsize=(10, 6))
    plt_x = x_lst_eval.T
    plt.axhline(y=65, color='red', linestyle='--', linewidth=2, label='65%, 70%, 75%')
    plt.axhline(y=70, color='red', linestyle='--', linewidth=2)
    plt.axhline(y=75, color='red', linestyle='--', linewidth=2)
    plt.plot(plt_x[0], all_train_acc*100, color='black', linestyle=':', linewidth=2, label='All Tempotrons (Cumulative)')
    plt.plot(plt_x[0], all_total_slices*100, color='black', linewidth=2, label='All Tempotrons')
    for i in range(10):
        plt.plot(plt_x[i], all_eval_slices.T[i]*100, linewidth=0.75, label='Tempotron '+str(i))
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.5)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("MNIST training sample (1 epoch = 1k spls)")
    plt.title("Accuracy across Training | 2 full epochs\nCombo "+str(param_combo)+" | Param ID: "+param_id)
    plt.tight_layout()
    #plt.savefig("TempotronNCC_MNIST_combo"+str(param_combo)+"_Acc_"+date_info+".pdf")
    plt.show()
    
    # save & plot final weights
    for n in range(10):
        dict_final_w['combo'+str(param_combo)]['Tempo'+str(n)] = list(tempotron_net.neurons[n].weights)

# Export all simulation final weight data to json output file
# file_path = "TempotronNCC_MNIST_finW_"+date_info+".json"
# with open(file_path, 'w') as json_file:
#     json.dump(dict_final_w, json_file, indent=4)
# print(f"Dictionary saved to {file_path}")


### Running Simulations for **Class-based Method** using *Sequential-Token Inputs (manually-derived scheme)*

In [ ]:
# Training all tempotrons for each pre-defined number of samples 
# try using TempotronNetwork_old vs TempotronNetwork

# update every time...will be used for plotting & saving data appropriately
date_info = "260713"

# Initialize basic parameters of interest
hint_options = {'hint_NA':None, 'hint_1':'hint_1'} 
lr_combo = {'combo1': [1, 0.001]}    # learning rate
w_m = 0.08                          # mean of weight initialization (from normal dist.)
w_sd = 0.025                         # standard deviation of weight initialization (from normal dist.)

#### training on all positions 0-4 (cycling through one long sequence)
training_set = df_train_options  # df_train_1k   #                     

# Define time window for each 4-state acc-memory sample simulation
t_enc = Tmax_nback_padded # ms
time_window = np.arange(0,t_enc,1)  

# Define number of epochs, total number of training samples, and how many often the network will be evaluated
# Note for running token-state sequence:
### structure: df_train_options and df_train_1k == {'seqs': [], 'possible_labels':[]}
epochs = 100 # each epoch trains on all 7 training spls
n_spls_1epoch = len(training_set['seqs'])
n_train_spls = n_spls_1epoch*epochs  # epoch max = 1000
n_eval_slice = n_spls_1epoch 

# useful list for tempotron output classes
output_classes = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

# Initialize dictionary to save final weights
dict_final_w = {}
dict_train_perform = {}
dict_train_history = {}

# For loop set-up to try out different learning rates associated with a unique experiment parameter-combo ID
for try_hint in hint_options:
    cx = 'combo1'
    param_combo = lr_combo[cx][0] 
    lr = lr_combo[cx][1] 
    param_id = 'w_m='+str(w_m)+', w_sd='+str(w_sd)+', lr='+str(lr)

    # Add dictionary to save final weights for this loop's lr_combo
    dict_final_w[cx+'_'+try_hint] = {'TempoA':[], 'TempoB':[], 'TempoC':[], 'TempoD':[], 'TempoE':[], 'TempoF':[], 'TempoG':[]}
    dict_train_history[cx+'_'+try_hint] = {k: {'sensitivity': [], 'specificity': [], 'f1': []} for k in range(len(output_classes))}
      
    # Create a Tempotron network with 7 neurons and 101 input channels
    add_in = 0
    if try_hint == 'hint_1':
        add_in = 1
    #tempotron_net = TempotronNetwork(n_neurons=7, n_inputs=numInputN+add_in, w_import_all=np.nan, w_mean=w_m, w_stdev=w_sd, time_window=time_window, tempo_labels=output_classes, lr=lr)
        # Note: to initialize weights from w_mean & w_stdev, use: w_import_all=np.nan ; otherwise, import weight array
    tempotron_net = TempotronNetwork(n_neurons=7, n_inputs=numInputN+add_in, w_import_all=np.nan, w_mean=w_m, w_stdev=w_sd, time_window=time_window, tempo_labels=output_classes, lr=lr)
    
    # Check to verify each tempotron object is a unique object
    #check_ids = [(id(t.V_peak), id(t.weights)) for t in tempotron_net.neurons]
    #print(len(set(check_ids)))

    # Initialize metric tracker
    tempotron_metrics = TempotronMetrics(n_classes=7)

    # Initialize arrays to store evaluation metrics
    all_train_acc = np.zeros((((n_train_spls//n_eval_slice)+1)))
    all_total_slices = np.zeros((((n_train_spls//n_eval_slice)+1)))
    all_eval_slices = np.zeros((((n_train_spls//n_eval_slice)+1),7))
    all_error_slices = np.zeros((((n_train_spls//n_eval_slice)+1),7))
    x_lst_eval = np.zeros((((n_train_spls//n_eval_slice)+1),7))
    
    ###
    # Begin Training
    ###
    seed_counter = 0
    for n_eval_epoch in range(n_train_spls):
        # Resets sample ID back to 0 for every epoch
        n_eval = n_eval_epoch % n_spls_1epoch

        # adding randomization/shuffling
        # Note: to have a better comparison against the alt methods, the training order should be standardized to the df_train_1k set
        if n_eval == 0:
            seed_counter += 1
            rng = np.random.default_rng(seed=seed_counter)
            arr_ind = np.arange(0,len(training_set['seqs']))
            shuffled_ind = rng.permutation(arr_ind)
            #print(shuffled_ind)
            tempotron_metrics.reset()
        n_shuf_ind = shuffled_ind[n_eval]

        # Generate spike trains & target output for single 4-state sequence
        spike_trains = build_inputspikes_general(training_set['seqs'][n_shuf_ind], numInputN, dict_spikepatterns_7tokens, 
                                                              [Tmax_nback_padded, t_btwn_state, time_per_state, 10, 1], if_plot=False, hint_type=hint_options[try_hint])
        spike_trains_T = spike_trains.T
        target_outputs = np.zeros(7)
        label_options = training_set['possible_labels'][n_shuf_ind]
        if len(label_options) == 1:
            target_outputs[output_classes.index(label_options[0])] = 1
            #print(label_options[0], output_classes.index(label_options[0]), target_outputs)
        else: # picks random option when there's more than one possible label
            rand_lab = np.random.choice(label_options)
            target_outputs[output_classes.index(rand_lab)] = 1
            #print(rand_lab, target_outputs)
        
        # Train step
        tempotron_net.train_step(spike_trains_T, target_outputs, metric_tracker=tempotron_metrics)
        
        # Evaluation pt. 1
        if (n_eval+1) % n_eval_slice == 0:
            
            x_lst_eval[(n_eval_epoch//n_eval_slice)+1][:] = n_eval_epoch
            all_train_acc[(n_eval_epoch//n_eval_slice)+1] = np.sum(tempotron_net.true_acc_log)/len(tempotron_net.true_acc_log[0])
            all_total_slices[(n_eval_epoch//n_eval_slice)+1] = np.sum([i[-n_eval_slice:] for i in tempotron_net.true_acc_log])/n_eval_slice
            
            
            rec_acc = np.sum([i[-n_eval_slice:] for i in tempotron_net.true_acc_log], axis=1)
            rec_tot = np.sum([i[-n_eval_slice:] for i in tempotron_net.all_targets], axis=1)
            all_eval_slices[(n_eval_epoch//n_eval_slice)+1] = rec_acc/rec_tot
            rec_err = np.sum([i[-n_eval_slice:] for i in tempotron_net.error_type_log], axis=1)
            all_error_slices[(n_eval_epoch//n_eval_slice)+1] = rec_err/n_eval_slice
            
            acc_str = ", ".join(f"{x:.2f}" for x in rec_acc/rec_tot)
            err_str = ", ".join(f"{x:.2f}" for x in rec_err/n_eval_slice)
            print(f"eval at {n_eval_epoch}, Training Acc: {acc_str}, Error {err_str}")#'eval at '+str(n_eval_epoch), rec_acc/rec_tot)
            
        # Evaluation pt. 2
        if (n_eval_epoch+1) % 100 == 0: # use this line when you want to run this evaluation less/more frequently
            epoch_metrics = tempotron_metrics.compute_confusion_metrics()
            y_evaluation = []
            y_predictions = []
            for jjj in range(len(training_set['seqs'])):
                # Generate spike trains & target output for single 4-state sequence
                spike_trains = build_inputspikes_general(training_set['seqs'][jjj], numInputN, dict_spikepatterns_7tokens, 
                                                        [Tmax_nback_padded, t_btwn_state, time_per_state, 10, 1], if_plot=False, hint_type=hint_options[try_hint])
                spike_trains_T = spike_trains.T
                target_outputs = np.zeros(7)
                label_options = training_set['possible_labels'][jjj]
                if len(label_options) == 1:
                    target_outputs[output_classes.index(label_options[0])] = 1
                    #print(label_options[0], output_classes.index(label_options[0]), target_outputs)
                else:
                    rand_lab = np.random.choice(label_options)
                    target_outputs[output_classes.index(rand_lab)] = 1
                    #print(rand_lab, target_outputs)
                y_evaluation.append(target_outputs)
                # Train step
                y_predictions.append(tempotron_net.predict(spike_trains_T))
                
            confusion_pure_m, confusion_all_m, no_decision_m, ambiguous_m = tempotron_metrics.evaluate_system(tempotron_net, y_predictions, y_evaluation)
            tempotron_metrics.plot_confusion_matrix(confusion_all_m, no_decision_m, ambiguous_m, epoch=seed_counter, class_names=output_classes, normalize=False, if_savefig=False, add_notes=cx+'_'+try_hint)
            for k in range(len(output_classes)):
                for key in ['sensitivity', 'specificity', 'f1']:
                    dict_train_history[cx+'_'+try_hint][k][key].append(epoch_metrics[k][key])
    ###
    # End of Training
    ###
    
    # Save & plot accuracy vs training progression 
    dict_train_perform[cx+'_'+try_hint] = {'true_acc_log': tempotron_net.true_acc_log, 
                                                                 'all_targets': tempotron_net.all_targets}

    # Plot & Save Accuracy across training
    plt.figure(figsize=(10, 6))
    plt_x = x_lst_eval.T
    plt.plot(plt_x[0], all_train_acc*100, color='black', linestyle=':', linewidth=2, label='All Tempotrons (Cumulative)')
    plt.plot(plt_x[0], all_total_slices*100, color='black', linewidth=2, label='All Tempotrons')
    for i in range(7):
        plt.plot(plt_x[i], all_eval_slices.T[i]*100, linewidth=0.75, label='Tempotron '+str(output_classes[i]))
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.5)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Tempotron | Training dataset (1 epoch = "+str(n_spls_1epoch)+" spls)")
    plt.title("Accuracy across Training \nCombo "+str(param_combo)+', '+try_hint+" | Param ID: "+param_id)
    plt.tight_layout()
    #plt.savefig("TempotronNCC_"+cx+'_'+try_hint+"_TrainAcc_"+date_info+".pdf")
    plt.show()

    # Plot & Save Error across training
    plt.figure(figsize=(10, 6))
    plt_x = x_lst_eval.T
    plt.axhline(y=12.5, color='red', linestyle='--', linewidth=2, label='12.5%')
    for i in range(7):
        plt.plot(plt_x[i], all_error_slices.T[i]*100, linewidth=0.75, label='Tempotron '+str(output_classes[i]))
    
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(axis='y', alpha=0.5)
    plt.ylabel("Error (%)")
    plt.xlabel("Tempotron | Training dataset (1 epoch = "+str(n_spls_1epoch)+" spls)")
    plt.title("Error across Training \nCombo "+str(param_combo)+', '+try_hint+" | Param ID: "+param_id)
    plt.tight_layout()
    #plt.savefig("TempotronNCC_"+cx+'_'+try_hint+"_TrainError_"+date_info+".pdf")
    plt.show()

    
    # save & plot final weights
    # for n, let in enumerate(output_classes):
    #     dict_final_w[cx+'_'+try_hint]['Tempo'+let] = list(tempotron_net.neurons[n].weights)

# Export all simulation final weight data to json output file
# file_path = "Tempotron_finW_"+date_info+".json"
# with open(file_path, 'w') as json_file:
#     json.dump(dict_final_w, json_file, indent=4)
# print(f"Dictionary saved to {file_path}")


### Running Simulations for **Function-based Method** using *Sequential-Token Input (manually-derived scheme)*

In [ ]:
# let's run a full simulation using (clunky) functions only...
# update every time...will be used for plotting & saving data appropriately
date_info = "260713"

# Initialize basic parameters of interest
hint_options = {'hint_NA':None, 'hint_1':'hint_1'}
tau_options = {'tau15':15}


for try_tau in tau_options:
    # Initialize final save object
    final_weight_matrix = {}
    final_error_progression = {}
    final_outcome_types = {}
    final_weights_LTtenperErr = {}
    
    for try_hint in hint_options:
        print(try_tau, try_hint)
        final_weight_matrix[try_hint] = {'TempoA':[], 'TempoB':[], 'TempoC':[], 'TempoD':[], 'TempoE':[], 'TempoF':[], 'TempoG':[]}
        final_error_progression[try_hint] = []
        final_outcome_types[try_hint] = {'TempoA':{}, 'TempoB':{}, 'TempoC':{}, 'TempoD':{}, 'TempoE':{}, 'TempoF':{}, 'TempoG':{}}
        final_weights_LTtenperErr[try_hint] = {'TempoA':{}, 'TempoB':{}, 'TempoC':{}, 'TempoD':{}, 'TempoE':{}, 'TempoF':{}, 'TempoG':{}}
        
        for str_temp in tempotron_options:
            _weights, _w_tracker, _error_prog, _outcome_type_count = train_nback_seqorder_tau(tempotron=str_temp, n_input=100, try_Tmax=Tmax_nback_padded, 
                                                                                              _tau=tau_options[try_tau], t_hint=try_hint, numTrainingSteps=1000, try_seed=32)
            final_weight_matrix[try_hint]['Tempo'+str_temp] = _weights.T
            final_error_progression[try_hint].append(_error_prog) #['Tempo'+str_temp] = _error_prog
            final_outcome_types[try_hint]['Tempo'+str_temp] = _outcome_type_count
            final_weights_LTtenperErr[try_hint]['Tempo'+str_temp] = _w_tracker
       
        plt_error_training(final_error_progression[try_hint], avg_window=100, title_notes='\n1k tr_spls, '+try_tau+', '+try_hint, savefig=False, savename='_1ktrspls_'+try_tau+'_'+try_hint+'_'+date_info)

    #Export objects to json files
    fin_weights_dict = {'hint_NA':{}, 'hint_1':{}} #final_weight_matrix
    fin_error_dict = {'hint_NA':{}, 'hint_1':{}}   #final_error_progression
    tr_outtypes_dict = {'hint_NA':{}, 'hint_1':{}} #final_outcome_types
    tr_weights_dict = {'hint_NA':{}, 'hint_1':{}}  #final_weights_LTtenperErr

    for k, k_hint in enumerate(['hint_NA', 'hint_1']):
        for i, i_tempo in enumerate(['TempoA', 'TempoB', 'TempoC', 'TempoD', 'TempoE', 'TempoF', 'TempoG']):
            fin_weights_dict[k_hint][i_tempo] = final_weight_matrix[k_hint][i_tempo].tolist()
            fin_error_dict[k_hint][i_tempo] = final_error_progression[k_hint][i].tolist()
            tr_outtypes_dict[k_hint][i_tempo] = {}
            tr_weights_dict[k_hint][i_tempo] = {}
            for _key1 in ['all']: 
                tr_outtypes_dict[k_hint][i_tempo][_key1] = {}
                for _key2 in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
                    tr_outtypes_dict[k_hint][i_tempo][_key1][_key2] = final_outcome_types[k_hint][i_tempo][_key1][_key2].tolist()
            for ppp in final_weights_LTtenperErr[k_hint][i_tempo].keys():
                tr_weights_dict[k_hint][i_tempo][ppp] = final_weights_LTtenperErr[k_hint][i_tempo][ppp].tolist() 
    
    # filepath_txt = ['trnW', 'trnET', 'finW', 'finE']
    # for j_ind, jj in enumerate([ tr_weights_dict, tr_outtypes_dict, fin_weights_dict, fin_error_dict]):
    #     file_path = "TempotronNCC_hint1vNA_"+try_tau+"_"+filepath_txt[j_ind]+"_"+date_info+".json"
    #     with open(file_path, 'w') as json_file:
    #         json.dump(jj, json_file, indent=4)
    #     print(f"Dictionary saved to {file_path}")
    

## Troubleshooting Efforts (so far)
---

### Notes from Gemini report:

- First major divergent point: The forward-passing PSP calculation
  - The class-based method miscalculates multi-spike voltages due to improper convolution. Spikes are summed and then convolved with the PSPkernel, thus, calculating K(t_1 + t_2).
  - The function-based method correctly calculates K(t_1) + K(t_2) by convolving the PSPkernel directly against the binary spike train.
- Second major divergent point: The backward-passing weight update
  - The class-based method correctly updates the weight after errors by isolating all valid spike times and applying the summed kernel-based change.
  - The function-based method incorrectly overwrites the history by updating the weight after errors where only the single most recent spike prior to t_max is applied in the kernel-based change.


In [ ]:
# Gemini evaluation of Method 1:
class TempotronNeuron_old:
    def __init__(self, n_inputs, w_import=False, w_mean=0.001, w_stdev=0.0001, tau=15.0, tau_s=3.75, v0=2.12, v_thresh=1.0, lr=0.001, label=''):
        self.n_inputs = n_inputs
        self.tau = tau
        self.tau_s = tau_s
        self.v0 = v0
        self.v_thresh = v_thresh
        self.lr = lr
        self.label = 'Tempotron_'+label
        self.reset_state()
        if np.isnan(w_import).all(): 
            self.weights = np.random.normal(w_mean, w_stdev, n_inputs)
        else:
            self.weights = w_import
            

    def reset_state(self):
        self.V_peak = 0.0
        self.V_trace = None

    def PSP_kernel(self, t_diff):
        """Post-synaptic potential kernel"""
        k = self.v0 * (np.exp(-t_diff / self.tau) - np.exp(-t_diff / self.tau_s))
        k[t_diff < 0] = 0
        return k

    def compute_voltage(self, spike_trains, time_window):
        all_K = np.zeros((self.n_inputs, len(time_window)))   
        for i in range(self.n_inputs):
            spikes = spike_trains[i]
            if np.sum(spikes) == 0:
                continue 
            t_diffs = np.convolve(spikes, time_window)[:len(time_window)] # GEMINI IDENTIFIED THIS LINE AS PROBLEMATIC
            all_K[i] = self.PSP_kernel(t_diffs)
        
        V = np.dot(self.weights, all_K)
        self.V_trace = V
        self.V_peak = np.max(V)
        return V

    def fire(self):
        return self.V_peak >= self.v_thresh

    def update_weights(self, spike_trains, time_window, target_output):
        pred = self.fire()
        if pred == target_output:
            return 0  # No error

        t_max = time_window[np.argmax(self.V_trace)]
        for i in range(self.n_inputs):
            spikes = spike_trains[i]
            if np.sum(spikes) == 0:
                continue
            t_diffs = t_max - np.array(np.where(spikes>0))
            valid_diffs = t_diffs[t_diffs >= 0] 
            if valid_diffs.size == 0:
                continue
            dV_dw = np.sum(self.PSP_kernel(valid_diffs))
            delta_w = self.lr * dV_dw
            if pred and not target_output:
                self.weights[i] -= delta_w
            elif not pred and target_output:
                self.weights[i] += delta_w
        self.reset_state()
        return 1  # Error occurred

class TempotronNetwork_old:
    def __init__(self, n_neurons, n_inputs, w_import_all, time_window, tempo_labels, **neuron_params):
        if np.isnan(w_import_all).all():
            self.neurons = [TempotronNeuron_old(n_inputs, w_import=[np.nan], label=tempo_labels[_], **neuron_params) for _ in range(n_neurons)]
        else:
            self.neurons = [TempotronNeuron_old(n_inputs, w_import=w_import_all[_], label=tempo_labels[_], **neuron_params) for _ in range(n_neurons)]
        self.time_window = time_window
        self.error_log = []
        self.error_type_log = []
        self.true_acc_log = []
        self.all_targets = []
        for i in self.neurons:
            self.error_type_log.append([])
            self.true_acc_log.append([])
            self.all_targets.append([])

    def reset(self):
        for neuron in self.neurons:
            neuron.reset_state()

    def train_step(self, spike_trains, target_outputs, metric_tracker=None):
        errors = []
        for n, neuron, target in zip(range(len(self.neurons)), self.neurons, target_outputs):
            #print(neuron.label, target)
            self.reset()
            neuron.compute_voltage(spike_trains, self.time_window)
            error = neuron.update_weights(spike_trains, self.time_window, target)
            errors.append(error)
            if metric_tracker != None:
                metric_tracker.update(n, error, target)
        self.error_log.append(np.sum(errors))
        for e, error in enumerate(errors):
            self.all_targets[e].append(target_outputs[e])
            self.error_type_log[e].append(error)
            self.true_acc_log[e].append(0)
        if np.sum(errors) == 0:
            if len(np.where(target_outputs > 0)) > 1:
                raise CustomError("Target output had more correct values than expected; see train_step function in TempotronNetwork.")
            if len(target_outputs) == 1:
                self.true_acc_log[0][-1] = 1
            else:
                self.true_acc_log[np.where(np.array(target_outputs) > 0)[0][0]][-1] = 1
            
        self.reset()

    def predict(self, spike_trains):
        preds = []
        for neuron in self.neurons:
            neuron.compute_voltage(spike_trains, self.time_window)
            preds.append(int(neuron.fire()))
        self.reset()
        return preds

In [ ]:
# Gemini evaluation of Method 2:
# Note MG: this code was modified slightly to remove notes and not reveal exact details about input structure to Gemini
def calcError_allseqs_tau(tempotron, weights, nn_input, try_Tmax, _tau=15.0, t_hint=None): 
    # setting evaluation dataset
    evaluation_seqs = df_train_options['seqs']
    evaluation_labels = df_train_options['possible_labels']

    # time series info
    if t_hint == 'hint_6':
        nn_input_hints = nn_input+14
    else:
        nn_input_hints = nn_input+4
    t_nback_padded = np.arange(0, try_Tmax, 1)
    v0 =  2.12
    _tau_s = _tau/4.0
    kernelT_padded = v0 * (np.exp(-t_nback_padded/_tau) - np.exp(-t_nback_padded/_tau_s))
    
    # setting theoretical limits 
    error_n = 0 
    type_template = {'TP':0, 'TN':0, 'FP':0, 'FN':0, 'stochastic_P':0, 'stochastic_N':0}
    count_outcome_type = {'Group 0':{}, 'Group 1':{}, 'Group 2':{}, 'Group 3':{}}
    for _key1 in count_outcome_type.keys():
        for _key2 in type_template.keys():
            count_outcome_type[_key1][_key2] = 0
    
    
    for a,b in enumerate(evaluation_seqs):
        pos_type = find_Pos_type(b)
        if len(evaluation_labels[a]) == 1:
            pattern_key = 1 if evaluation_labels[a][0] == tempotron else -1
        else:
            pattern_key = 0 
        spikePattern = build_inputspikes_general(seqN, n_input) # exact same input applied across methods
        spikePatternT = spikePattern.copy().T
        
        K = np.zeros((nn_input_hints, t_nback_padded.size)) 
    
        for j in range(nn_input_hints):
            K[j,:] = np.convolve(spikePatternT[j, :], kernelT_padded)[:t_nback_padded.size]
    
        tempotronPSP_ind = np.dot(weights, K)[0] 
        idx_max = np.argmax(tempotronPSP_ind)
        v_max = tempotronPSP_ind[idx_max]
    
        if ( v_max > 1 and pattern_key < 0 ):
            error_n += 1
            count_outcome_type[pos_type]['FP'] += 1
        elif ( v_max < 1 and pattern_key > 0 ):
            error_n += 1
            count_outcome_type[pos_type]['FN'] += 1
        elif ( v_max < 1 and pattern_key < 0 ):
            count_outcome_type[pos_type]['TN'] += 1
        elif ( v_max > 1 and pattern_key > 0 ):
            count_outcome_type[pos_type]['TP'] += 1
        elif ( v_max > 1 and pattern_key == 0 ):
            count_outcome_type[pos_type]['stochastic_P'] += 1
            if tempotron=='G' or tempotron=='C' or tempotron=='F':
                error_n += 1
        elif ( v_max < 1 and pattern_key == 0 ):
            count_outcome_type[pos_type]['stochastic_N'] += 1
        else:
            raise ValueError('i dont even know lol')

    error_prop = error_n/len(evaluation_seqs)
    return error_prop, count_outcome_type

def train_nback_seqorder_tau(tempotron, n_input, try_Tmax, _tau=15.0, t_hint=None, numTrainingSteps=50000, try_seed=32): #prev. function input set-up: ...(numBatches, target, training_style)
    training_seqorder = df_train_1k['seqs']
    training_labelorder = df_train_1k['possible_labels']

    if t_hint == 'hint_6':
        n_input_hints = n_input+14
    else:
        n_input_hints = n_input+4
    learningRate = 0.005
    
    #initialize weights 
    np.random.seed(try_seed) #hold at seed = 32 for all methods
    weights = np.random.normal(loc=0.1, scale=(0.025), size= (1, n_input_hints))

    # time-series info
    t_nback_padded = np.arange(0, try_Tmax, 1)
    v0  = 2.12
    _tau_s = _tau/4.0
    kernelT_padded = v0 * (np.exp(-t_nback_padded/_tau) - np.exp(-t_nback_padded/_tau_s))
    
    #Markers for training efficiency
    weights_tracker = {} 
    error_progression = np.zeros((numTrainingSteps+1))
    otype_template = {'TP':0, 'TN':0, 'FP':0, 'FN':0, 'stochastic_P':0, 'stochastic_N':0}
    outcome_type_count = {'Group 0':{}, 'Group 1':{}, 'Group 2':{}, 'Group 3':{}}
    for _key1 in ['Group 0', 'Group 1', 'Group 2', 'Group 3']:
        for _key2 in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
            outcome_type_count[_key1][_key2] = np.zeros((numTrainingSteps+1))
    
    error_progression[0], current_outcome = calcError_allseqs_tau(tempotron, weights, n_input, try_Tmax, _tau, t_hint) 
    for _key1 in ['Group 0', 'Group 1', 'Group 2', 'Group 3']:
        for _key2 in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
            outcome_type_count[_key1][_key2][0] = current_outcome[_key1][_key2]
    
    for i in range(numTrainingSteps):
        
        #initialize K; will be used later to calculate tempotron response to spike pattern
        K = np.zeros((n_input_hints, try_Tmax)) 

        #select pair from training sequence & assign true label
        seqN = training_seqorder[i]      
        labelN = training_labelorder[i]

        #grab/call appropriate spike pattern 
        spikePattern = build_inputspikes_general(seqN, n_input) # exact same input applied across methods
        spikePatternT = spikePattern.copy().T
        
        if labelN == tempotron:
            pattern_key = 1
            
        else:
            pattern_key = -1
        
        for j in range(n_input_hints):        
            K[j,:] =  np.convolve(spikePatternT[j, :], kernelT_padded)[:t_nback_padded.size] 
        tempotronPSP = np.dot(weights, K)[0]
        idx_max = np.argmax(tempotronPSP)
        v_max = tempotronPSP[idx_max]

    
        if ( v_max > 1 and pattern_key < 0 ) or ( v_max < 1 and pattern_key > 0 ):
                #enters if loop when: (1) fires and was not supposed to OR (2) did not fire and was supposed to
            wi_update = np.zeros(n_input_hints)
            t_last_spike = np.zeros((n_input_hints))

             
            for t in t_nback_padded[:idx_max]: 
                for k in range(n_input_hints): 
                    if spikePatternT[k,t]==1: 
                        t_last_spike[k] = t+1 # GEMINI IDENTIFIED THIS LINE AS PROBLEMATIC
                         
            for k in range(n_input_hints):
                if t_last_spike[k] > 0:
                    diff = t_nback_padded[idx_max] -  (t_last_spike[k]-1)  
                    
                    if diff > 0:
                        wi_update[k] += v0 * (np.exp(-diff/_tau) - np.exp(-diff/_tau_s)) 
            
            if pattern_key < 0:
                weights -= learningRate*wi_update
            elif pattern_key > 0:
                weights += learningRate*wi_update
                
        
        error_progression[i+1], current_outcome  = calcError_allseqs_tau(tempotron, weights, n_input, try_Tmax, _tau, t_hint) 
       
        for hh in ['Group 0', 'Group 1', 'Group 2', 'Group 3']:
            for jj in ['TP', 'TN', 'FP', 'FN', 'stochastic_P', 'stochastic_N']:
                outcome_type_count[hh][jj][i+1] = current_outcome[hh][jj]

        if error_progression[i+1] <= 0.10:
            weights_tracker[i] = weights

        if error_progression[i+1] == 0.0000 and error_progression[i] == 0.0000:
            return weights, weights_tracker, error_progression, outcome_type_count

    return weights, weights_tracker, error_progression, outcome_type_count

### Evaluation Idea

Track and plot weight changes every step (same seed) across class-based and function-based codes. Do they diverge? where?